<a href="https://colab.research.google.com/github/soleildayana/Celestial-Mechanics/blob/main/2BodiesEnElTiempo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cuaderno de clase
## Mecánica Celeste (2026-1) con Jorge I. Zuluaga 30 Abr 2026
## El problema de los dos cuerpos en el tiempo

In [1]:
!pip install -Uq pymcel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.9 MB/s eta 0:00:00


In [2]:
import pymcel as pc
import numpy as np
deg = np.pi / 180

Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!


In [3]:
e	= 0.1911663355386932
a	= 0.9223803173917017 * pc.constantes.au
q	= 0.7460522521429133 * pc.constantes.au
i	= 3.340958441017069*deg
node = 203.8996515621043*deg
peri = 126.6728325163065*deg
tp = 2461042.918242006079 * 86400 # fecha juliana
mu = pc.constantes.mu_sun # Este es el mu de todo el sistema solar

Varaibles derivadas:

In [4]:
h = np.sqrt(mu * a * (1 - e**2))
b = a * np.sqrt(1 - e**2)

h, b

(np.float64(4200387442699322.5), np.float64(135441343761.16716))

¿Cuál es el tiempo?

In [5]:
from astropy.time import Time

In [6]:
t = Time("2029-04-13 18:52:00", scale='tdb').jd * 86400
t

np.float64(212737560720.0)

Escribir la ecuación en la forma: f(E) = 0

In [7]:
def ecuacion_kepler(E):
  funcion = E - e*np.sin(E) - h/(a*b)*(t-tp)
  return funcion

Resolver la ecuación de Kepler:

In [8]:
from scipy.optimize import newton

In [9]:
E = newton(ecuacion_kepler, 0)
E

np.float64(23.081594014937842)

Fórmulas aproximadas
$$ E = M + esin(M)  $$
 $$  E = M + esin(M) + e²/2sin(2M)$$

In [18]:
M = h/(a*b)*(t-tp)

E1 = M + e*np.sin(M)
print('Anomalía excéntrica de primer orden: ', E1)

E2 = M + e*np.sin(M) + e**2/2*np.sin(2*M)
print('Anomalía excéntrica de segundo orden: ', E2)

E3 = M + e*np.sin(M) + e**2/2*np.sin(2*M) + e**3/8*(3*np.sin(3*M) - np.sin(M))
print('Anomalía excéntrica de tercer orden: ', E3)

E4 = M + e*np.sin(M) + e**2/2*np.sin(2*M) + e**3/8*(3*np.sin(3*M) - np.sin(M)) + e**4/24*(2*np.sin(4*M) - np.sin(2*M))
print('Anomalía excéntrica de cuarto orden: ', E4)

E = E4

Anomalía excéntrica de primer orden:  23.069120261548683
Anomalía excéntrica de segundo orden:  23.07976142540877
Anomalía excéntrica de tercer orden:  23.082153879075918
Anomalía excéntrica de cuarto orden:  23.08201609662143


In [15]:
E = newton(ecuacion_kepler, 0)
E

np.float64(23.081594014937842)

In [21]:
# Rutina pymcel

anomalia, error, pasos = pc.kepler_newton(M,e, delta = 1e-15) # Devuleve el número de recurrencias

print(f'se obtuvo una anomalía de {anomalia} +- {error} en {pasos} pasos.')

se obtuvo una anomalía de 23.081594014937842 +- 0.0 en 6 pasos.


Obtenemos ahora la anomalía verdadera:

In [22]:
f = 2*np.arctan(np.sqrt((1+e)/(1-e))*np.tan(E/2))
f

np.float64(-2.2142097119138278)

Una vez tenemos la anomalía verdadera podemos calcular la posición en su sistema perifocal:

In [23]:
p = a*(1-e**2)
r = p / (1 + e * np.cos(f))
xf = r * np.cos(f) # Perifocales
yf = r * np.sin(f)
zf = 0

Usando la rotación en el espacio:

In [24]:
import spiceypy as spy
R = spy.eul2m(-node, -i, -peri, 3, 1, 3)

r = R @ np.array([xf, yf, zf])
r

array([-1.37492488e+11, -6.03777060e+10, -2.93317563e+07])

Vamos a comparar la posición con la que nos da consulta horizons:


In [25]:
tabla, jd, X = pc.consulta_horizons(id='Apophis', location='@SSB', epochs='2029-04-13 18:52:00')
X[:3]

/usr/local/lib/python3.12/dist-packages/erfa/core.py:133: ErfaWarning: ERFA function "dtf2d" yielded 1 of "dubious year (Note 6)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


array([-1.37263651e+11, -6.04834602e+10, -4.52634041e+06])

In [ ]:
e	= 0.1911663355386932
a	= 0.9223803173917017 * pc.constantes.au
q	= 0.7460522521429133 * pc.constantes.au
i	= 3.340958441017069*deg
node = 203.8996515621043*deg
peri = 126.6728325163065*deg
tp = 2461042.918242006079 * 86400 # fecha juliana
mu = pc.constantes.mu_sun # Este es el mu de todo el sistema solar

In [26]:
spy.conics([q,e,i,node,peri,M,tp,mu],t)


array([-9.34271501e+10,  1.30555013e+11, -9.17751835e+09, -2.27345265e+04,
       -1.31133413e+04,  1.62193263e+02])